In [3]:
import pandas as pd

df = pd.read_parquet("all_images_8.parquet")

print(df.head())
print(df.shape)

                                               image  \
0  /data/test_images/2022_02_02 5nm Ag nanopartic...   
1  /data/test_images/2022_02_02 5nm Ag nanopartic...   
2  /data/test_images/2022_02_02 5nm Ag nanopartic...   
3  /data/test_images/2022_02_02 5nm Ag nanopartic...   
4  /data/test_images/2022_02_02 5nm Ag nanopartic...   

                                           clustered  
0  [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...  
1  [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, ...  
2  [0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...  
3  [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, ...  
4  [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, ...  
(13, 2)


In [4]:
import glob
import os
from ncempy.io import dm
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.preprocessing import StandardScaler
import umap
from sklearn.cluster import KMeans
from skimage.util import view_as_windows
import numpy as np
from persim import PersistenceImager
from ripser import lower_star_img
import pandas as pd
from itertools import islice

/home/eng-6770/Documents/TEM_Exploration/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def get_positions(image_data, patch_size=64, stride=16):
    if stride is None:
        stride = patch_size

    blocks = view_as_windows(
        image_data,
        (patch_size, patch_size),
        step=stride
    )

    n_rows, n_cols = blocks.shape[:2]

    positions = [
        (i * stride, j * stride)
        for i in range(n_rows)
        for j in range(n_cols)
    ]

    return positions

def label_to_image(original_image, labels, positions, patch_size=64):
    H, W = original_image.shape[:2]

    label_img = np.zeros((H, W), dtype=np.uint8)
    vote_img  = np.zeros((H, W), dtype=np.uint8)

    for (r, c), lab in zip(positions, labels):
        r_end = min(r + patch_size, H)
        c_end = min(c + patch_size, W)

        region = (slice(r, r_end), slice(c, c_end))

        votes = vote_img[region] + 1

        mask = votes >= vote_img[region]

        label_patch = label_img[region]
        label_patch[mask] = lab

        vote_img[region][mask] = votes[mask]

    return label_img

In [29]:
import numpy as np
from skimage.morphology import remove_small_objects
from scipy.ndimage import binary_fill_holes
from skimage.morphology import binary_closing, disk
from pathlib import Path

image_dir = Path("../hrtem_files/2022_02_02 5nm Ag nanoparticles on UTC/")
labels_dir = Path("../hrtem_files/2022_02_02 5nm Ag nanoparticles on UTC/Labels")

output = Path("./cluster_results")


for id, row in df.iterrows():

    image_name = Path(row["image"]).stem
    out_dir = output / image_name
    out_dir.mkdir(parents=True, exist_ok=True)  

    matches = list(image_dir.glob(f"{image_name}.dm3"))
    truths = list(labels_dir.glob(f"{image_name}_label.png"))

    if not matches:
        continue

    image_path = matches[0]
    truth_path = truths[0]
    data_dict = dm.dmReader(str(Path(image_path).resolve()))
    original_image = data_dict['data']
    truth_img = mpimg.imread(truth_path).astype(np.uint8)

    labels = row["clustered"]
    positions = get_positions(original_image, 64, 16)
    label_img = label_to_image(original_image, row["clustered"], positions)
    output_path = out_dir / "labelimg.png"

   # plt.imsave(output_path, label_img, cmap="tab20")

    clusters = [0,1,2,3,4,5,6,7]

    for i in clusters:

        pred_mask  = (label_img == i)        
        truth_mask = (truth_img > 0)  

        clean_mask = remove_small_objects(pred_mask, max_size=100)
        clean_mask = binary_fill_holes(clean_mask)
        clean_mask = binary_closing(clean_mask, footprint=disk(2))
        clean_mask = remove_small_objects(clean_mask, max_size=1600)

        intersection = (clean_mask & truth_mask).sum()
        union        = (clean_mask | truth_mask).sum()

        iou  = intersection / union                              

        print(f"Image {str(image_name)} Cluster {i} — IoU: {iou:.3f} ")

        #plt.imsave(out_dir / f"cluster_{i}.png", mask, cmap="tab20")



/tmp/ipykernel_805888/3605568842.py:47: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  clean_mask = binary_closing(clean_mask, footprint=disk(2))


Image 20220202_Ag_UTC_330kx_2650e_0p1596s_08 Cluster 0 — IoU: 0.014 


/tmp/ipykernel_805888/3605568842.py:47: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  clean_mask = binary_closing(clean_mask, footprint=disk(2))


Image 20220202_Ag_UTC_330kx_2650e_0p1596s_08 Cluster 1 — IoU: 0.035 


/tmp/ipykernel_805888/3605568842.py:47: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  clean_mask = binary_closing(clean_mask, footprint=disk(2))


...

Image 20220202_Ag_UTC_330kx_2650e_0p1596s_11 Cluster 5 — IoU: 0.012 


/tmp/ipykernel_805888/3605568842.py:47: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  clean_mask = binary_closing(clean_mask, footprint=disk(2))


Image 20220202_Ag_UTC_330kx_2650e_0p1596s_11 Cluster 6 — IoU: 0.017 


/tmp/ipykernel_805888/3605568842.py:47: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  clean_mask = binary_closing(clean_mask, footprint=disk(2))


Image 20220202_Ag_UTC_330kx_2650e_0p1596s_11 Cluster 7 — IoU: 0.013 


Raw masks
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_08 Cluster 4 — IoU: 0.174 
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_05 Cluster 7 — IoU: 0.248 
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_07 Cluster 2 — IoU: 0.103 
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_03 Cluster 6 — IoU: 0.392 
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_02 Cluster 1 — IoU: 0.347 
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_10 Cluster 1 — IoU: 0.276 
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_13 Cluster 1 — IoU: 0.318 
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_12 Cluster 6 — IoU: 0.374 
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_14 Cluster 1 — IoU: 0.225 
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_09 Cluster 2 — IoU: 0.245 
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_04 Cluster 3 — IoU: 0.300 
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_06 Cluster 2 — IoU: 0.267 
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_11 Cluster 4 — IoU: 0.288

Cleaned masks:
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_08 Cluster 4 — IoU: 0.191 (+ 0.017)
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_05 Cluster 7 — IoU: 0.272 (+ 0.024)
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_07 Cluster 2 — IoU: 0.101 (- 0.002)
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_03 Cluster 6 — IoU: 0.392 (=)
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_02 Cluster 1 — IoU: 0.354 (+ 0.007)
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_10 Cluster 1 — IoU: 0.291 (+ 0.015)
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_13 Cluster 1 — IoU: 0.331 (+ 0.013)
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_12 Cluster 6 — IoU: 0.378 (+ 0.004)
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_14 Cluster 1 — IoU: 0.230 (+ 0.005)
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_09 Cluster 2 — IoU: 0.276 (+ 0.031)
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_04 Cluster 3 — IoU: 0.332 (+ 0.032)
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_06 Cluster 2 — IoU: 0.277 (+ 0.010)
Image 20220202_Ag_UTC_330kx_2650e_0p1596s_11 Cluster 4 — IoU: 0.321 (+ 0.033)